In [1]:
import pandas as pd
import torch
import sys

In [2]:
def choose_device():
    if torch.cuda.is_available():
        return "cuda"
    # if torch.backends.mps.is_available():
    #     return "mps"
    return "cpu"

In [3]:
device = choose_device()
device

'cpu'

In [4]:
%load_ext autoreload
%autoreload 2
    
sys.path.append("FMs/BulkFormer/")
from BulkFormer import load_bulkformer, bulkformer_embed

/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: dlopen(/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch_scatter/_version_cpu.so, 0x0006): Symbol not found: __ZN5torch3jit17parseSchemaOrNameERKNSt3__112basic_stringIcNS1_11char_traitsIcEENS1_9allocatorIcEEEEb
  Referenced from: <A0916C8D-111C-3C4A-BA0C-223427DB5BB3> /Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch_scatter/_version_cpu.so
  Expected in:     <699F2277-8EEE-3C07-BE7E-3B893740EE04> /Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: dlo

In [5]:
patient = pd.read_csv('prep/dataset/tcga/gene_exp.csv.gz', index_col=0)
patient = patient.drop('cases.submitter_id', axis=1)
patient.head()

,ENSG00000121410,ENSG00000156006,ENSG00000196839,ENSG00000170558,ENSG00000117020,ENSG00000275199,ENSG00000236362,ENSG00000242551,ENSG00000256628,ENSG00000133997,...,ENSG00000135164,ENSG00000154845,ENSG00000039068,ENSG00000140199,ENSG00000119314,ENSG00000159197,ENSG00000070413,ENSG00000288475,ENSG00000118412,ENSG00000284194
0,0.000000,0.074458,2.524624,4.648279,3.209694,3.209694,0.0,0.000000,0.785909,2.795175,...,3.071174,3.013184,0.016169,2.545759,2.695559,0.089475,4.139910,2.270919,2.270919,0.134356
1,0.040950,0.159735,2.061201,0.245922,1.021767,1.021767,0.0,0.054678,1.361361,2.324444,...,2.294805,2.764676,0.009059,2.039166,3.122902,0.099755,3.189451,1.796399,1.796399,0.077516
2,0.320706,0.275205,3.170244,0.624440,2.880333,2.880333,0.0,0.211071,1.737866,2.754959,...,2.968623,3.607488,0.158285,2.023308,3.293111,0.000000,3.715981,1.710640,1.710640,0.798003
6,0.030820,1.080550,3.319405,3.084155,2.461467,2.461467,0.0,0.225221,2.134616,2.838751,...,2.536771,3.594876,0.059683,1.759202,3.315382,0.000000,4.512946,1.681666,1.681666,0.544879
7,0.042101,0.201389,2.094909,0.474991,1.562724,1.562724,0.0,0.000000,1.114748,3.266263,...,2.288141,2.360052,0.018625,2.382182,2.745982,0.499926,4.222852,1.627180,1.627180,0.222423


In [6]:
bf_obj = load_bulkformer(device=device)

In [7]:
%%time
emb = bulkformer_embed(
    log_tpm_df=patient,
    bf_obj=bf_obj,
    feature_type="transcriptome_level",
    aggregate_type="mean",   # "max" や "all" に変えてもOK
    batch_size=32,
    return_expr_value=False,
)

print(emb.shape)  # [N_samples, D]

[BulkFormer] Covered genes: 19357 / 20010 (96.74%)
[BulkFormer] Missing genes : 653 (filled with -10)


100%|████████████████████████████████████████████████████████████████████████████████████| 138/138 [6:03:38<00:00, 158.11s/it]

torch.Size([4416, 640])
CPU times: user 23h 53min 7s, sys: 20h 31min 56s, total: 1d 20h 25min 4s
Wall time: 6h 3min 39s


In [18]:
torch.save(emb, "Embs/patient_exp_emb.pt")
emb.shape

torch.Size([4416, 640])

In [9]:
celline = pd.read_csv('prep/dataset/gdsc/exp.csv.gz', index_col=0)
celline.head()

,ENSG00000121410,ENSG00000268895,ENSG00000148584,ENSG00000175899,ENSG00000245105,ENSG00000166535,ENSG00000256661,ENSG00000256904,ENSG00000256069,ENSG00000184389,...,ENSG00000203995,ENSG00000232242,ENSG00000162378,ENSG00000159840,ENSG00000274572,ENSG00000074755,ENSG00000036549,ENSG00000167524,ENSG00000253251,ENSG00000272899
451Lu,0.430580,2.554713,0.0,5.505329,0.227066,0.027930,0.000000,0.00000,0.000000,0.227066,...,0.055101,0.313501,3.199025,4.968280,0.0,2.848262,2.789104,1.573539,1.525426,1.506769
A388,0.293037,0.072939,0.0,0.000000,0.173249,2.631951,0.000000,0.00000,0.037134,0.037134,...,0.173249,0.000000,3.131355,5.309036,0.0,4.107183,3.561062,3.822491,2.252586,0.758454
A427,0.630893,1.774765,0.0,0.030920,3.109818,0.000000,0.090032,0.03092,0.000000,0.030920,...,2.502331,0.000000,2.670040,4.898954,0.0,2.191095,2.739346,1.774765,2.468327,0.893859
A431,0.128913,0.968466,0.0,0.000000,0.187630,0.113679,0.000000,0.05031,0.000000,0.143919,...,0.243090,0.000000,3.288343,4.742639,0.0,2.081585,3.361940,2.232623,1.246069,0.553629
ACN,0.640576,2.659146,0.0,4.769836,0.306771,0.035273,0.102294,0.00000,0.000000,0.332846,...,0.541195,0.919455,4.163511,4.823200,0.0,2.536277,3.664105,2.928531,1.831273,1.053250


In [10]:
%%time
cl_emb = bulkformer_embed(
    log_tpm_df=celline,
    bf_obj=bf_obj,
    feature_type="transcriptome_level",
    aggregate_type="mean",   # "max" や "all" に変えてもOK
    batch_size=32,
    return_expr_value=False,
)

print(cl_emb.shape)  # [N_samples, D]

[BulkFormer] Covered genes: 19194 / 20010 (95.92%)
[BulkFormer] Missing genes : 816 (filled with -10)


100%|████████████████████████████████████████████████████████████████████████████████████████| 14/14 [36:13<00:00, 155.25s/it]

torch.Size([442, 640])
CPU times: user 2h 22min 12s, sys: 2h 4min 19s, total: 4h 26min 31s
Wall time: 36min 13s


In [14]:
torch.save(cl_emb, "Embs/cl_exp_emb.pt")
cl_emb.shape

torch.Size([442, 640])